# Graph-construction transfer (v5-ct-2026-10-05)
**Before running:** Session options → **GPU T4 x2**, **Internet On**; Input → attach the dataset containing **`construction_transfer_code_v5.zip`**.
Run cell 1 by hand to check the version, then **Save Version → Save & Run All**. Download `results_v5.zip` from Output.

In [ ]:
# 1. Copy the code from the attached dataset
import glob, os, shutil, zipfile
EXPECTED = 'v5-ct-2026-10-05'
W = '/kaggle/working/ct'
for z in glob.glob('/kaggle/input/**/construction_transfer_code_v5.zip', recursive=True):
    zipfile.ZipFile(z).extractall('/kaggle/tmp_ct')
versions = glob.glob('/kaggle/input/**/CODE_VERSION.txt', recursive=True) + glob.glob('/kaggle/tmp_ct/**/CODE_VERSION.txt', recursive=True)
found = {v: open(v).read().strip() for v in versions}
print('code versions found:', found)
good = [v for v, ver in found.items() if ver == EXPECTED]
assert good, f'Attach the dataset with construction_transfer_code_v5.zip (version {EXPECTED}). Found: {found}'
shutil.copytree(os.path.dirname(good[0]), W, dirs_exist_ok=True)
os.chdir(W)
print('using code from:', os.path.dirname(good[0]))
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install PyTorch Geometric (keeps Kaggle's GPU PyTorch) and download the datasets once.
#    Downloads are retried (servers sometimes time out); a dataset that still fails is skipped, not fatal.
!pip install -q torch_geometric
import torch, time, shutil
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on Accelerator = GPU T4 x2'
from run import load
AVAILABLE, FAILED = [], []
for name in ('cora', 'citeseer', 'pubmed', 'photo', 'computers', 'cs', 'wikics'):
    for attempt in range(5):
        try:
            X, y, _ = load(name, 'data')
            print(name, X.shape, 'classes', int(y.max()) + 1)
            AVAILABLE.append(name)
            break
        except Exception as e:
            print(f'{name}: download attempt {attempt + 1} failed ({type(e).__name__}: {e})')
            if name == 'wikics':
                shutil.rmtree('data/WikiCS', ignore_errors=True)   # clear a half-finished download
            time.sleep(30 * (attempt + 1))
    else:
        FAILED.append(name)
print('available:', AVAILABLE)
print('SKIPPED (download failed 5 times):', FAILED or 'none')

In [ ]:
# 3. Checks: unit tests (expect 53 passed) + model forward passes + a smoke test of the v5 pipeline
!python -m pytest -q tests
!python models.py
!rm -rf /tmp/smoke && python run_v5.py --datasets cora --models gcn sage --seeds 0 --out /tmp/smoke/v5_s.csv 2>&1 | grep -v -i warn | tail -3
import pandas as pd
_s = pd.read_csv('/tmp/smoke/v5_s.csv'); _g = pd.read_csv('/tmp/smoke/v5_s_graphs.csv')
assert len(_s) == 2 * len(_g) ** 2 and _s[['acc_test', 'agree_est', 'atc_est', 'doc_est']].notna().all().all(), 'smoke test failed'
print('SMOKE TEST OK:', len(_g), 'graphs,', len(_s), 'rows')
print(_g[['recipe', 'h1', 'h2', 'avg_degree', 'h_node_sd', 'sim_same', 'sim_cross']].round(3).to_string(index=False))

In [ ]:
# 4. Run on both GPUs (one commit), printing progress live.
import subprocess, time
os.makedirs('results', exist_ok=True)
DEADLINE = str(int(time.time() + 10.5 * 3600))   # no new (dataset, seed, model) unit starts after 10.5 h
GROUPS = {0: ['cora', 'citeseer', 'photo', 'computers'], 1: ['pubmed', 'cs', 'wikics']}
GROUPS = {g: [d for d in ds if d in AVAILABLE] for g, ds in GROUPS.items()}
print('running:', GROUPS, '| skipped:', FAILED)
jobs = {g: ['--datasets', *ds, '--out', f'results/v5_gpu{g}.csv'] for g, ds in GROUPS.items() if ds}
procs, logs = {}, {}
for gpu, a in jobs.items():
    logs[gpu] = f'results/gpu{gpu}.log'
    procs[gpu] = subprocess.Popen(['python', 'run_v5.py', *a], stdout=open(logs[gpu], 'w'), stderr=subprocess.STDOUT,
                                  env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), DEADLINE=DEADLINE))
seen = {g: 0 for g in jobs}
while True:
    for g in jobs:
        lines = open(logs[g]).read().splitlines()
        for line in lines[seen[g]:]:
            if 'Warning' not in line:
                print(f'[gpu{g}] {line}', flush=True)
        seen[g] = len(lines)
    if all(p.poll() is not None for p in procs.values()):
        break
    time.sleep(30)
failed = [g for g, p in procs.items() if p.returncode != 0]
if failed:
    print(f'WARNING: GPU {failed} stopped with an error (see [gpu] lines above). Results that finished are still saved and zipped below.')
else:
    print('done')

In [ ]:
# 5. Analysis + results zip (download results_v5.zip from the Output tab)
!python analyze_v5.py "results/v5_gpu*.csv" --outdir results/analysis_v5 2>&1 | grep -v -i warn
!cd /kaggle/working && zip -qr results_v5.zip ct/results && ls -lh results_v5.zip